## Modelo baseline de Machine Learning

In [0]:
# carregamento dos dados para o modelo de Machine Learning
df_origem = spark.table("workspace.default.manufacturing_defect_dataset")

print("Dataset carregado com sucesso!")
print(f"Total de registros: {df_origem.count()}")

## 1. Preparação dos dados para o modelo
O objetivo desta etapa é preparar os dados sem treinar o modelo ainda.

In [0]:
# bibliotecas para Machine Learning
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# variáveis utilizadas como entrada
colunas_modelo = [
    "ProductionVolume",
    "ProductionCost",
    "SupplierQuality",
    "DeliveryDelay",
    "MaintenanceHours",
    "DowntimePercentage",
    "InventoryTurnover",
    "StockoutRate",
    "WorkerProductivity",
    "SafetyIncidents",
    "EnergyConsumption",
    "EnergyEfficiency",
    "AdditiveProcessTime",
    "AdditiveMaterialCost"
]

# converte os dados para Pandas
df_ml = df_origem.select(colunas_modelo + ["DefectStatus"]).toPandas()

# separa variáveis de entrada e variável-alvo
X = df_ml[colunas_modelo]
y = df_ml["DefectStatus"]

# separação estratificada: 80% treino e 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Preparação concluída!")
print("Registros de treinamento:", len(X_train))
print("Registros de teste:", len(X_test))

print("\nDistribuição das classes no treinamento:")
print(y_train.value_counts())

print("\nDistribuição das classes no teste:")
print(y_test.value_counts())

## 2. Treinamento do Random Forest

O Random Forest é um algoritmo que combina várias árvores de decisão para produzir uma classificação. É uma boa escolha porque consegue identificar relações entre diferentes indicadores sem exigir uma configuração muito complexa.

In [0]:
# cria o modelo
modelo = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)

# treina o modelo
modelo.fit(X_train, y_train)

# realiza previsões
y_pred = modelo.predict(X_test)

print("Modelo treinado com sucesso!")

## 3. Cálculo das métricas

Nesta etapa, será avaliado se o modelo está conseguindo classificar corretamente os registros.

In [0]:
# calcula as métricas
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

# matriz de confusão
matriz = confusion_matrix(y_test, y_pred)

print(f"Precisão (Precision): {precision:.3f}")
print(f"Recall: {recall:.3f}")

print("\nMatriz de confusão:")
print(matriz)

print(f"\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))

In [0]:
from sklearn.dummy import DummyClassifier

# cria o modelo de referência
modelo_dummy = DummyClassifier(strategy="most_frequent")

# treina o modelo 
modelo_dummy.fit(X_train, y_train)
# realiza as previsões
y_pred_dummy = modelo_dummy.predict(X_test)

# calcula a precisão
precision_dummy = precision_score(y_test, y_pred_dummy, zero_division=0)
# calcula o recall
recall_dummy = recall_score(y_test, y_pred_dummy, zero_division=0)

print(f"Dummy Precision: {precision_dummy:.3f}")
print(f"Dummy Recall: {recall_dummy:.3f}")

print("\nMatriz de confusão do Dummy:")
print(confusion_matrix(y_test, y_pred_dummy))

In [0]:
# salva o modelo treinado
joblib.dump(modelo, "modelo.joblib")

print("Modelo treinado foi salvo!")

In [0]:
# TESTE DE INTEGRAÇÃO DO MODELO
# carrega o modelo salvo
modelo_carregado = joblib.load("modelo.joblib")

# faz previsões com os dados de teste
previsoes = modelo_carregado.predict(X_test)

print("Modelo foi carregado com sucesso!")
print("Primeiras 15 previsões: ", previsoes[:15])

O teste acima demonstra que não é preciso treinar o modelo novamente para utilizá-lo. Podemos carregar o arquivo salvo e fazer novas previsões.